# BrotherTrue: verificação do decoder próprio

Fontes isoladas, pesos inicializados do zero. Este notebook executa regressões pequenas no Colab e salva o relatório no Drive. Não treina um modelo comercial. O ZIP deve estar na mesma pasta do Drive que este notebook.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, shutil, sys, tempfile, zipfile
drive.mount('/content/drive')
if sys.platform != 'linux' or not Path('/content').is_dir():
    raise RuntimeError('Google Colab required')
base = Path('/content/drive/MyDrive/BrotherTrue_experimental_2026-10-26')
bundle = base / 'brothertrue_source_bundle.zip'
expected_sha256 = 'a50832d7ee0ded27e03c8db24d5f945c53fdec793b9e81cfb3e4078ccda7df1b'
if hashlib.sha256(bundle.read_bytes()).hexdigest() != expected_sha256:
    raise RuntimeError('Source ZIP hash mismatch')
if shutil.disk_usage('/content').free < 4.5 * 1024**3:
    raise RuntimeError('Colab disk below 4.5 GiB')
work = Path(tempfile.mkdtemp(prefix='brothertrue_', dir='/content'))
with zipfile.ZipFile(bundle) as archive:
    for item in archive.infolist():
        target = (work / item.filename).resolve()
        if work not in target.parents or item.file_size > 4 * 1024**2:
            raise RuntimeError('Invalid bundle entry')
    archive.extractall(work)
manifest = json.loads((work / 'SOURCE_MANIFEST.json').read_text())
for name, meta in manifest['files'].items():
    if hashlib.sha256((work/name).read_bytes()).hexdigest() != meta['sha256']:
        raise RuntimeError('Source hash mismatch: ' + name)
os.environ['HF_HUB_OFFLINE'] = '1'
sys.path.insert(0, str(work))
experiment = work / 'experimentos/industrial_readiness_2026_10_25'
sys.path.insert(0, str(experiment))
print('Verified sources:', len(manifest['files']), 'workdir:', work)


In [ ]:
import subprocess, time
import torch
torch.set_num_threads(2)
results = []
for filename in ('test_contracts.py', 'test_runner_contract.py', 'test_own_voice.py'):
    started = time.monotonic()
    result = subprocess.run([sys.executable, '-B', str(experiment/'tests'/filename)],
                            cwd=work, capture_output=True, text=True)
    print(result.stdout, result.stderr)
    results.append({'suite':filename, 'exit_code':result.returncode,
                    'elapsed_seconds':time.monotonic()-started,
                    'stdout':result.stdout, 'stderr':result.stderr})
report = {'tests':results, 'torch_version':torch.__version__,
          'cuda_available':torch.cuda.is_available(),
          'gpu_name':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
          'source_zip_sha256':expected_sha256, 'training_executed':False,
          'verification_passed':all(r['exit_code'] == 0 for r in results),
          'commercial_readiness':'not_established'}
fd, temp_report = tempfile.mkstemp(prefix='verification_', suffix='.json', dir=base)
with os.fdopen(fd, 'w') as handle:
    json.dump(report, handle, indent=2)
report_path = base / (Path(temp_report).stem + '_complete.json')
os.replace(temp_report, report_path)
print('Saved verification:', report_path)
if not report['verification_passed']:
    raise RuntimeError('Verification failed; evidence preserved; no training or promotion')


## Próxima etapa

Após testes aprovados: produzir traços reais do núcleo; revisar corpus com professores; congelar treino/validação/teste; medir custo do pré-treinamento; executar piloto do `training/own_decoder_runner.py` conforme sua CLI. O checkpoint do aluno inclui todos os pesos. Os arquivos históricos de SFT Qwen não são o caminho de treinamento deste aluno.
